# Module 0: SFT vs. Reinforcement Learning — Two Ways to Train an Agent

## Learning Objectives

By completing this notebook, you will:
1. Understand what SFT data looks like and why imitation alone is insufficient for agents
2. Implement a reward function that scores agent outputs without labeled examples
3. Trace through a multi-step agent loop and see where SFT's coverage problem appears
4. Compute relative advantages across a group of attempts — the core idea behind GRPO

## Prerequisites
- Basic Python (lists, dicts, functions)
- Familiarity with what a language model does at a high level

## Estimated Time: 15 minutes

---

## Setup

This notebook only needs `numpy` and `matplotlib` — no API keys required. Every cell runs end-to-end on a standard Python environment.

In [ ]:
# Purpose: All imports in one place
# Key Concept: We deliberately avoid ML frameworks here — the ideas are clearer in pure Python

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import sqlite3
import re
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional

np.random.seed(42)  # Reproducibility — every run produces the same results

print("Setup complete.")

---

## Part 1: What SFT Data Looks Like

Supervised fine-tuning (SFT) trains a model by **imitation**: given an input, produce the exact output in the training set. The model minimises cross-entropy loss against a reference answer — it has no other signal.

For a text-to-SQL agent, an SFT dataset is a list of `(natural-language question, correct SQL)` pairs curated by humans. Let's build one.

In [ ]:
# Purpose: Represent an SFT training example and a small dataset
# Key Concept: SFT requires a SINGLE correct output per input — there is no notion of "almost right"

@dataclass
class SFTExample:
    """
    One supervised fine-tuning training pair.

    Parameters
    ----------
    question : str
        Natural-language question from the user
    gold_sql : str
        The single correct SQL query a human annotator wrote
    schema : str
        Table/column context the model must use
    """
    question: str
    gold_sql: str
    schema: str


SCHEMA = """
Tables:
  orders(order_id, customer_id, amount, status, created_at)
  customers(customer_id, name, region, tier)
""".strip()

# A realistic SFT dataset — small but representative
sft_dataset: List[SFTExample] = [
    SFTExample(
        question="How many orders were placed in 2024?",
        gold_sql="SELECT COUNT(*) FROM orders WHERE created_at >= '2024-01-01' AND created_at < '2025-01-01';",
        schema=SCHEMA,
    ),
    SFTExample(
        question="What is the total revenue from premium customers?",
        gold_sql="SELECT SUM(o.amount) FROM orders o JOIN customers c ON o.customer_id = c.customer_id WHERE c.tier = 'premium';",
        schema=SCHEMA,
    ),
    SFTExample(
        question="List the top 5 regions by order count.",
        gold_sql="SELECT c.region, COUNT(*) AS order_count FROM orders o JOIN customers c ON o.customer_id = c.customer_id GROUP BY c.region ORDER BY order_count DESC LIMIT 5;",
        schema=SCHEMA,
    ),
    SFTExample(
        question="Which customers have pending orders?",
        gold_sql="SELECT DISTINCT c.name FROM customers c JOIN orders o ON c.customer_id = o.customer_id WHERE o.status = 'pending';",
        schema=SCHEMA,
    ),
    SFTExample(
        question="What is the average order amount per region?",
        gold_sql="SELECT c.region, AVG(o.amount) AS avg_amount FROM orders o JOIN customers c ON o.customer_id = c.customer_id GROUP BY c.region;",
        schema=SCHEMA,
    ),
]

print(f"SFT dataset: {len(sft_dataset)} examples\n")
print(f"Example 0:")
print(f"  Question : {sft_dataset[0].question}")
print(f"  Gold SQL : {sft_dataset[0].gold_sql}")

### The SFT Coverage Problem

SFT looks clean until you ask: *what happens at inference time when the model generates something different from the gold answer, even if that answer is also correct?*

SQL has many valid phrasings for the same query. The cell below shows equivalent queries for Example 0 that an SFT model would penalise as wrong — even though they return identical results.

In [ ]:
# Purpose: Demonstrate that SFT's single-gold-answer assumption is a hard constraint
# Key Concept: Correct != same-as-gold. SFT cannot distinguish these two failure modes.

gold = sft_dataset[0].gold_sql

# These all return the same rows — but SFT scores them identically to gibberish
semantically_equivalent_sql = [
    "SELECT COUNT(order_id) FROM orders WHERE YEAR(created_at) = 2024;",
    "SELECT COUNT(*) FROM orders WHERE created_at BETWEEN '2024-01-01' AND '2024-12-31';",
    "SELECT COUNT(1) FROM orders WHERE strftime('%Y', created_at) = '2024';",
]

print("Gold answer (SFT target):")
print(f"  {gold}\n")

print("Semantically equivalent — but SFT gives these ZERO credit:")
for i, sql in enumerate(semantically_equivalent_sql, 1):
    # SFT loss is token-level cross-entropy: not-gold == maximum loss
    token_match = sql.strip() == gold.strip()
    print(f"  [{i}] {sql}")
    print(f"       SFT credit: {'FULL' if token_match else 'ZERO'}")

print("\nSFT has no way to reward 'close but not identical'.")
print("This is the coverage problem: the model learns one path, not the goal.")

---

## Part 2: The RL Alternative — Score the Outcome, Not the Path

Reinforcement learning replaces the gold label with a **reward function**: any output that achieves the goal gets a positive reward, regardless of how it arrived there.

For SQL generation the goal is clear: does the query execute without error AND return the right rows? We can write that check directly in Python.

In [ ]:
# Purpose: Build an in-memory database so we can actually execute SQL and check results
# Key Concept: The reward is grounded in real execution — not string comparison

def build_test_database() -> sqlite3.Connection:
    """
    Create an in-memory SQLite database with orders and customers tables.

    Returns
    -------
    sqlite3.Connection
        Connection to the populated in-memory database
    """
    conn = sqlite3.connect(":memory:")
    cur = conn.cursor()

    cur.executescript("""
        CREATE TABLE customers (
            customer_id INTEGER PRIMARY KEY,
            name        TEXT,
            region      TEXT,
            tier        TEXT
        );

        CREATE TABLE orders (
            order_id    INTEGER PRIMARY KEY,
            customer_id INTEGER,
            amount      REAL,
            status      TEXT,
            created_at  TEXT
        );

        INSERT INTO customers VALUES
            (1, 'Apex Corp',   'North', 'premium'),
            (2, 'Beta LLC',    'South', 'standard'),
            (3, 'Gamma Inc',   'North', 'premium'),
            (4, 'Delta Co',    'East',  'standard'),
            (5, 'Epsilon Ltd', 'West',  'premium');

        INSERT INTO orders VALUES
            (101, 1, 1500.0, 'completed', '2024-03-15'),
            (102, 2,  800.0, 'pending',   '2024-07-22'),
            (103, 3, 2200.0, 'completed', '2024-11-01'),
            (104, 4,  300.0, 'completed', '2023-12-28'),
            (105, 5, 3100.0, 'pending',   '2024-06-10'),
            (106, 1,  950.0, 'completed', '2024-09-03'),
            (107, 3, 1750.0, 'cancelled', '2024-02-14');
    """)

    conn.commit()
    return conn


DB = build_test_database()

# Sanity check
rows = DB.execute("SELECT COUNT(*) FROM orders;").fetchone()
print(f"Database ready: {rows[0]} orders, ", end="")
rows = DB.execute("SELECT COUNT(*) FROM customers;").fetchone()
print(f"{rows[0]} customers.")

In [ ]:
# Purpose: Implement a reward function that scores SQL responses against a live database
# Key Concept: Reward = execution success + result correctness. No labels needed.

def score_sql_attempt(
    generated_sql: str,
    reference_sql: str,
    conn: sqlite3.Connection,
) -> Tuple[float, str]:
    """
    Score a generated SQL query against a reference result.

    Scoring rubric:
      +0.0  — syntax error (query does not execute)
      +0.3  — executes but returns wrong rows
      +0.6  — returns correct row COUNT but wrong values
      +1.0  — returns identical result set to reference

    Parameters
    ----------
    generated_sql : str
        SQL produced by the agent
    reference_sql : str
        A known-correct SQL query that defines the expected result
    conn : sqlite3.Connection
        Database to execute queries against

    Returns
    -------
    Tuple[float, str]
        (reward in [0, 1], human-readable explanation)
    """
    # Step 1: Try to execute the generated SQL
    try:
        gen_rows = conn.execute(generated_sql).fetchall()
    except sqlite3.Error as exc:
        return 0.0, f"Syntax/execution error: {exc}"

    # Step 2: Execute reference to get ground truth
    ref_rows = conn.execute(reference_sql).fetchall()

    # Step 3: Check result equality (order-insensitive)
    if sorted(gen_rows) == sorted(ref_rows):
        return 1.0, "Correct: identical result set"

    # Step 4: Partial credit — right count, wrong values
    if len(gen_rows) == len(ref_rows):
        return 0.6, f"Partial: right count ({len(ref_rows)} rows) but wrong values"

    # Step 5: Executes but completely off
    return 0.3, f"Wrong: got {len(gen_rows)} rows, expected {len(ref_rows)}"


# Demonstrate the reward function on the three equivalent queries from Part 1
reference = sft_dataset[0].gold_sql  # Known-correct anchor

test_queries = [
    ("Gold SQL (SFT target)",       reference),
    ("BETWEEN variant",             "SELECT COUNT(*) FROM orders WHERE created_at BETWEEN '2024-01-01' AND '2024-12-31';"),
    ("strftime variant",            "SELECT COUNT(1) FROM orders WHERE strftime('%Y', created_at) = '2024';"),
    ("Wrong column",                "SELECT COUNT(*) FROM orders WHERE amount > 100;"),
    ("Syntax error",                "SELEKT COUNT(*) FORM orders;"),
]

print(f"Question: {sft_dataset[0].question}\n")
print(f"{'Label':<30} {'Reward':>8}  Reason")
print("-" * 72)
for label, sql in test_queries:
    reward, reason = score_sql_attempt(sql, reference, DB)
    print(f"{label:<30} {reward:>8.1f}  {reason}")

**What changed?** The `BETWEEN` and `strftime` variants now receive full reward. RL does not care *how* the model arrived at the correct result — only that it did.

This is the key distinction:
- **SFT**: Minimise distance to the gold path
- **RL**: Maximise expected reward from the environment

---

## Part 3: The Multi-Step Problem — Why SFT Breaks for Agents

Real agents do not produce a single output. They chain tool calls: inspect the schema, draft a query, check its result, refine if needed. Each step depends on the previous one.

SFT would need a gold transcript for *every possible execution path*. That is impossible to collect at scale. RL only needs the final reward — the intermediate steps can vary freely as long as the outcome is correct.

In [ ]:
# Purpose: Implement a 3-step agent loop that chains tool calls
# Key Concept: The agent loop produces a trajectory; RL rewards the FINAL state

@dataclass
class AgentStep:
    """
    One step in an agent trajectory.

    Parameters
    ----------
    tool : str
        Name of the tool called
    input : str
        Argument passed to the tool
    output : str
        Result returned by the tool
    """
    tool: str
    input: str
    output: str


@dataclass
class AgentTrajectory:
    """
    Full sequence of steps taken by the agent for one question.

    Parameters
    ----------
    question : str
        Original user question
    steps : List[AgentStep]
        Ordered list of tool calls made
    final_sql : str
        The SQL query submitted as the final answer
    reward : float
        Score assigned by the reward function (set after execution)
    """
    question: str
    steps: List[AgentStep] = field(default_factory=list)
    final_sql: str = ""
    reward: float = 0.0


# ---- Tool implementations ------------------------------------------------

def tool_inspect_schema(table_name: str, conn: sqlite3.Connection) -> str:
    """
    Return column names and types for a table.

    Parameters
    ----------
    table_name : str
        Table to inspect
    conn : sqlite3.Connection
        Active database connection

    Returns
    -------
    str
        Column description string
    """
    try:
        rows = conn.execute(f"PRAGMA table_info({table_name});").fetchall()
        cols = ", ".join(f"{r[1]} ({r[2]})" for r in rows)
        return f"{table_name}: [{cols}]"
    except Exception as exc:
        return f"Error inspecting {table_name}: {exc}"


def tool_run_sql(sql: str, conn: sqlite3.Connection) -> str:
    """
    Execute SQL and return a preview of the result.

    Parameters
    ----------
    sql : str
        Query to execute
    conn : sqlite3.Connection
        Active database connection

    Returns
    -------
    str
        Result preview or error message
    """
    try:
        rows = conn.execute(sql).fetchall()
        return f"{len(rows)} row(s): {rows[:3]}{'...' if len(rows) > 3 else ''}"
    except sqlite3.Error as exc:
        return f"SQL error: {exc}"


def tool_verify_result(result_preview: str, expected_description: str) -> str:
    """
    Simple heuristic check — in production this would call the LLM.

    Parameters
    ----------
    result_preview : str
        Output from tool_run_sql
    expected_description : str
        Natural-language description of what the result should contain

    Returns
    -------
    str
        Verification verdict
    """
    if "SQL error" in result_preview:
        return "FAIL: query did not execute"
    if "0 row(s)" in result_preview:
        return "WARN: result is empty — check WHERE clause"
    return f"OK: result looks plausible for '{expected_description}'"


# ---- Agent loop ----------------------------------------------------------

def run_agent(
    question: str,
    draft_sql: str,
    conn: sqlite3.Connection,
    reference_sql: str,
) -> AgentTrajectory:
    """
    Execute a 3-step agent: inspect schema, run draft SQL, verify result.

    Parameters
    ----------
    question : str
        User question driving the agent
    draft_sql : str
        SQL the agent's policy produced
    conn : sqlite3.Connection
        Live database for tool calls
    reference_sql : str
        Ground-truth SQL used by the reward function

    Returns
    -------
    AgentTrajectory
        Full trajectory including reward
    """
    traj = AgentTrajectory(question=question)

    # Step 1 — inspect schema to ground the query
    schema_info = tool_inspect_schema("orders", conn)
    traj.steps.append(AgentStep("inspect_schema", "orders", schema_info))

    # Step 2 — run the draft SQL
    run_result = tool_run_sql(draft_sql, conn)
    traj.steps.append(AgentStep("run_sql", draft_sql, run_result))

    # Step 3 — verify the result looks right
    verify_result = tool_verify_result(run_result, question)
    traj.steps.append(AgentStep("verify_result", run_result, verify_result))

    # Assign final SQL and compute reward
    traj.final_sql = draft_sql
    traj.reward, _ = score_sql_attempt(draft_sql, reference_sql, conn)

    return traj


# Run the agent with two different drafts and compare trajectories
example = sft_dataset[0]

traj_good  = run_agent(example.question, example.gold_sql, DB, example.gold_sql)
traj_alt   = run_agent(
    example.question,
    "SELECT COUNT(*) FROM orders WHERE created_at BETWEEN '2024-01-01' AND '2024-12-31';",
    DB, example.gold_sql,
)
traj_wrong = run_agent(
    example.question,
    "SELECT COUNT(*) FROM orders WHERE status = 'completed';",
    DB, example.gold_sql,
)

for label, traj in [("Gold SQL", traj_good), ("Equivalent alt", traj_alt), ("Wrong query", traj_wrong)]:
    print(f"--- Trajectory: {label} ---")
    for i, step in enumerate(traj.steps, 1):
        print(f"  Step {i} [{step.tool}]")
        print(f"    in : {step.input[:80]}")
        print(f"    out: {step.output[:80]}")
    print(f"  Final reward: {traj.reward:.1f}\n")

The two correct trajectories take *different intermediate paths* (different SQL variants) but both earn reward = 1.0. An SFT dataset would need a human-written transcript of every valid path. RL sidesteps this entirely: it only labels the trajectory *endpoint*.

---

## Part 4: Group Scoring Preview — A Taste of GRPO

Group Relative Policy Optimisation (GRPO) — the algorithm at the core of this course — generates **G** attempts at the same question, scores each one, then reinforces attempts that scored *above the group mean* and suppresses those that scored *below*.

This "relative advantage" framing is more stable than absolute rewards because it automatically calibrates to the current policy's performance level.

In [ ]:
# Purpose: Generate 4 candidate SQL attempts, score each, compute relative advantages
# Key Concept: Advantage = reward - group_mean. Positive => reinforce. Negative => suppress.

question  = sft_dataset[1].question   # "What is the total revenue from premium customers?"
reference = sft_dataset[1].gold_sql

# Four plausible attempts that a policy at different training stages might produce
# (In real GRPO these come from sampling the model at temperature > 0)
attempts = [
    {
        "label": "Attempt A",
        "description": "Correct join + WHERE on tier",
        "sql": (
            "SELECT SUM(o.amount) "
            "FROM orders o "
            "JOIN customers c ON o.customer_id = c.customer_id "
            "WHERE c.tier = 'premium';"
        ),
    },
    {
        "label": "Attempt B",
        "description": "Subquery variant — also correct",
        "sql": (
            "SELECT SUM(amount) FROM orders "
            "WHERE customer_id IN ("
            "  SELECT customer_id FROM customers WHERE tier = 'premium'"
            ");"
        ),
    },
    {
        "label": "Attempt C",
        "description": "Joins correctly but sums wrong column",
        "sql": (
            "SELECT SUM(o.order_id) "
            "FROM orders o "
            "JOIN customers c ON o.customer_id = c.customer_id "
            "WHERE c.tier = 'premium';"
        ),
    },
    {
        "label": "Attempt D",
        "description": "Missing JOIN — wrong table reference",
        "sql": "SELECT SUM(amount) FROM orders WHERE tier = 'premium';",
    },
]

# Score every attempt
for attempt in attempts:
    reward, reason = score_sql_attempt(attempt["sql"], reference, DB)
    attempt["reward"] = reward
    attempt["reason"] = reason

rewards = np.array([a["reward"] for a in attempts])

# GRPO relative advantage: normalise rewards around the group mean
group_mean = rewards.mean()
group_std  = rewards.std() + 1e-8   # epsilon avoids division by zero
advantages = (rewards - group_mean) / group_std

for attempt, adv in zip(attempts, advantages):
    attempt["advantage"] = adv

# Print the scoring table
print(f"Question: {question}\n")
print(f"{'Label':<12} {'Reward':>8} {'Advantage':>12}  Decision")
print("-" * 58)
for a in attempts:
    decision = "REINFORCE" if a["advantage"] > 0 else "SUPPRESS "
    print(
        f"{a['label']:<12} {a['reward']:>8.1f} {a['advantage']:>12.3f}"
        f"  {decision}  ({a['description']})"
    )
print(f"\nGroup mean reward: {group_mean:.3f}  |  Group std: {group_std:.3f}")

In [ ]:
# Purpose: Visualise the group scoring — which attempts get reinforced vs. suppressed
# Key Concept: Positive advantage (above mean) => policy gradient pushes these up in probability

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

labels = [a["label"] for a in attempts]
rwd    = [a["reward"]    for a in attempts]
adv    = [a["advantage"] for a in attempts]
colors = ["#2ecc71" if a > 0 else "#e74c3c" for a in adv]

# ---- Left: raw rewards --------------------------------------------------
ax = axes[0]
bars = ax.bar(labels, rwd, color=colors, edgecolor="white", linewidth=1.2)
ax.axhline(group_mean, color="#3498db", linewidth=2, linestyle="--", label=f"Group mean = {group_mean:.2f}")
ax.set_ylim(0, 1.25)
ax.set_ylabel("Reward", fontsize=12)
ax.set_title("Raw Reward per Attempt", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(axis="y", alpha=0.3)
for bar, val in zip(bars, rwd):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.02, f"{val:.1f}",
            ha="center", va="bottom", fontsize=11, fontweight="bold")

# ---- Right: relative advantages -----------------------------------------
ax = axes[1]
bars = ax.bar(labels, adv, color=colors, edgecolor="white", linewidth=1.2)
ax.axhline(0, color="black", linewidth=1.5)
ax.set_ylabel("Normalised Advantage", fontsize=12)
ax.set_title("Relative Advantage (GRPO Signal)", fontsize=13, fontweight="bold")
ax.grid(axis="y", alpha=0.3)
for bar, val in zip(bars, adv):
    offset = 0.05 if val >= 0 else -0.12
    ax.text(bar.get_x() + bar.get_width() / 2, val + offset, f"{val:+.2f}",
            ha="center", va="bottom", fontsize=11, fontweight="bold")

reinforce_patch = mpatches.Patch(color="#2ecc71", label="Reinforce (above mean)")
suppress_patch  = mpatches.Patch(color="#e74c3c", label="Suppress (below mean)")
axes[1].legend(handles=[reinforce_patch, suppress_patch], fontsize=10)

fig.suptitle(f'GRPO Group Scoring\n"{question}"', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

print("\nKey insight: Attempt B is ALSO reinforced despite using a subquery (not the gold SQL).")
print("RL rewards the outcome, not the approach.")

### What GRPO Does With These Advantages

In Module 1 you will derive this fully. For now, the intuition:

- **Positive advantage** (A, B): the policy gradient update *increases* the log-probability of producing these tokens next time
- **Negative advantage** (C, D): the update *decreases* their probability
- No external labels needed — the reward function does the labelling at runtime

---

## Part 5: Exercise — Classify and Implement

This exercise has two independent parts. **Modify the code** — do not just run the existing cells.

### Exercise 0.1: SFT or RL? Classify Four Scenarios

**Task:** For each scenario below, set the value in `classifications` to either `"SFT"` or `"RL"` and fill in `reasons` with a one-sentence justification. Run the test cell to check your answers.

**Scenarios:**
1. Train a model to translate English sentences into French using 1M human-translated pairs
2. Train an agent to write Python code that passes a unit test suite — the test results are the only signal
3. Train a chatbot to match a specific company's writing style using 500 example support tickets
4. Train an agent to play chess — the only feedback is win/loss at the end of the game

**Hints:**
<details>
<summary>Hint 1</summary>
Ask: does the training signal come from (a) matching a human-labelled output, or (b) evaluating the final result?
</details>

<details>
<summary>Hint 2</summary>
SFT requires a gold output for every input. If you cannot write down the "correct" output before seeing the model's attempt, you need RL.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------
# Replace None with "SFT" or "RL" for each scenario
# Replace the empty strings with your one-sentence reasons

classifications = {
    "scenario_1_translation":    None,   # <-- "SFT" or "RL"
    "scenario_2_code_tests":     None,
    "scenario_3_writing_style":  None,
    "scenario_4_chess":          None,
}

reasons = {
    "scenario_1_translation":    "",     # <-- your justification
    "scenario_2_code_tests":     "",
    "scenario_3_writing_style":  "",
    "scenario_4_chess":          "",
}

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------

def test_exercise_0_1():
    CORRECT = {
        "scenario_1_translation":   "SFT",
        "scenario_2_code_tests":    "RL",
        "scenario_3_writing_style": "SFT",
        "scenario_4_chess":         "RL",
    }

    valid = {"SFT", "RL"}

    for scenario, expected in CORRECT.items():
        answer = classifications.get(scenario)
        assert answer is not None, (
            f"classifications['{scenario}'] is still None. "
            "Set it to 'SFT' or 'RL'."
        )
        assert answer in valid, (
            f"classifications['{scenario}'] = {answer!r} is not valid. "
            "Use exactly 'SFT' or 'RL'."
        )
        assert answer == expected, (
            f"Incorrect for {scenario}: you said {answer!r}, expected {expected!r}.\n"
            "Hint: does training require a pre-labelled gold output (SFT) "
            "or only a reward signal evaluated after the fact (RL)?"
        )

    for scenario in CORRECT:
        reason = reasons.get(scenario, "")
        assert len(reason.strip()) >= 10, (
            f"Please add a justification for '{scenario}' "
            "(at least 10 characters)."
        )

    print("Exercise 0.1 passed! Your classifications:")
    for s, c in classifications.items():
        print(f"  {s:<35} => {c}  | {reasons[s]}")

test_exercise_0_1()

### Exercise 0.2: Implement a Reward Function for a Different Task

**Task:** Implement `score_python_attempt` — a reward function for a **code-writing agent** whose task is to produce Python functions that pass a given set of unit tests.

The rubric:
- `0.0` — the code raises a `SyntaxError` or `NameError` when loaded
- `0.5` — the code loads without error but fails at least one test
- `1.0` — the code passes all provided tests

**Expected output:** Running the test cell below should print `Exercise 0.2 passed!`

**Hints:**
<details>
<summary>Hint 1</summary>
Use Python's built-in `compile()` to check syntax, then `exec()` inside a try/except to load the code into a local namespace.
</details>

<details>
<summary>Hint 2</summary>
After exec-ing the code into a namespace dict, call each test function from that namespace and count how many raise an AssertionError.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

def score_python_attempt(
    generated_code: str,
    test_functions: List,
) -> Tuple[float, str]:
    """
    Score a generated Python function against a list of test callables.

    Parameters
    ----------
    generated_code : str
        Python source code produced by the agent (defines one or more functions)
    test_functions : list of callable
        Each callable accepts a namespace dict and returns None (passes) or
        raises AssertionError (fails)

    Returns
    -------
    Tuple[float, str]
        (reward in {0.0, 0.5, 1.0}, explanation)
    """
    # Step 1: Check for syntax errors
    # YOUR CODE HERE

    # Step 2: Execute the code into a local namespace
    # YOUR CODE HERE

    # Step 3: Run each test function; count failures
    # YOUR CODE HERE

    return 0.0, "Not implemented yet"  # replace this line


your_reward_fn = score_python_attempt  # Do not rename this variable

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------

def test_exercise_0_2():
    assert callable(your_reward_fn), "score_python_attempt must be a callable"

    # --- Test fixtures ---
    # The agent's task: implement a function called `add` that returns a + b

    def test_add_positive(ns):
        assert ns["add"](2, 3) == 5, "add(2, 3) should return 5"

    def test_add_negative(ns):
        assert ns["add"](-1, -1) == -2, "add(-1, -1) should return -2"

    def test_add_zero(ns):
        assert ns["add"](0, 0) == 0, "add(0, 0) should return 0"

    tests = [test_add_positive, test_add_negative, test_add_zero]

    # Case 1: Syntax error
    broken_syntax = "def add(a, b) return a + b"   # missing colon
    reward, reason = your_reward_fn(broken_syntax, tests)
    assert reward == 0.0, (
        f"Syntax-error code should receive reward 0.0, got {reward}.\n"
        "Check that you return 0.0 when compile() raises SyntaxError."
    )

    # Case 2: Correct implementation
    correct_code = "def add(a, b):\n    return a + b\n"
    reward, reason = your_reward_fn(correct_code, tests)
    assert reward == 1.0, (
        f"Correct implementation should receive reward 1.0, got {reward}.\n"
        "Make sure all test functions run against the exec'd namespace."
    )

    # Case 3: Code loads but fails tests
    wrong_code = "def add(a, b):\n    return a - b\n"  # subtraction, not addition
    reward, reason = your_reward_fn(wrong_code, tests)
    assert reward == 0.5, (
        f"Code that fails tests should receive reward 0.5, got {reward}.\n"
        "Return 0.5 when the code executes but at least one test assertion fails."
    )

    # Case 4: Reward is numeric
    assert isinstance(reward, float), "Reward must be a float"

    print("Exercise 0.2 passed!")
    print(f"  Syntax error => reward {your_reward_fn(broken_syntax, tests)[0]:.1f}")
    print(f"  Wrong logic  => reward {your_reward_fn(wrong_code, tests)[0]:.1f}")
    print(f"  Correct code => reward {your_reward_fn(correct_code, tests)[0]:.1f}")

test_exercise_0_2()

---

## Summary

**Key takeaways from this notebook:**

1. **SFT trains by imitation.** It requires a gold-labelled output per input and penalises any deviation, including semantically correct alternatives. This creates a coverage problem for tasks with multiple valid solutions.

2. **RL trains by outcome.** The reward function replaces the gold label. Any output that achieves the goal — regardless of path — receives positive reward. This naturally handles the multi-solution problem.

3. **Agents compound the SFT coverage problem.** A 3-step tool-calling chain has an exponential number of valid trajectories. SFT would need to label all of them. RL only needs to score the final state.

4. **GRPO uses group-relative advantages.** Instead of absolute rewards, it normalises across G samples from the same prompt. Attempts above the group mean get reinforced; those below get suppressed. This is more stable than absolute reward baselines.

5. **The reward function IS the specification.** Writing a good reward function is the most important design decision in RL for agents — as you saw in Exercise 0.2.

---

**What's next:**
- **Module 1** — GRPO algorithm in depth: the policy gradient derivation, the group sampling procedure, and why the KL penalty prevents reward hacking
- **Module 2** — The ART framework: how to structure agent roles, tools, and reward shaping for multi-step tasks

**Additional reading:**
- DeepSeek-R1 technical report (introduces GRPO in the reasoning-model context)
- Schulman et al., "Proximal Policy Optimization Algorithms" (2017) — PPO, GRPO's ancestor
- Ouyang et al., "Training language models to follow instructions with human feedback" (2022) — RLHF baseline